In [4]:
import requests
import pandas as pd

BASE_URL = "https://ghoapi.azureedge.net/api/"

INDICATORS = {
    "HWF_0001": "Hekim Yogunlugu",
    "HWF_0006": "Hemsire-Ebe Yogunlugu",
    "WHS4_100": "DTP3 Asi Orani",
}

REGION_NAMES = {
    "AFR": "Africa",
    "AMR": "Americas",
    "SEAR": "South-East Asia",
    "EUR": "Europe",
    "EMR": "Eastern Mediterranean",
    "WPR": "Western Pacific",
}

YEAR_START = 2019
YEAR_END = 2023

C:\Users\Eser\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
def fetch_country_lookup() -> dict:
    """Ulke kodu (ISO3) -> ulke adi eslemesini ceker."""
    resp = requests.get(BASE_URL + "DIMENSION/COUNTRY/DimensionValues", timeout=30)
    resp.raise_for_status()
    values = resp.json()["value"]
    return {v["Code"]: v["Title"] for v in values}

country_lookup = fetch_country_lookup()
print(f"Toplam ulke sayisi: {len(country_lookup)}")
print(dict(list(country_lookup.items())[:5]))  # ilk 5 ornek

Toplam ulke sayisi: 234
{'ABW': 'Aruba', 'AFG': 'Afghanistan', 'AGO': 'Angola', 'AIA': 'Anguilla', 'ALB': 'Albania'}


In [6]:
def fetch_indicator(indicator_code: str) -> list:
    """Bir gostergenin tum ulke verisini ceker, sayfalama varsa takip eder."""
    url = f"{BASE_URL}{indicator_code}"
    params = {
        "$filter": (
            f"SpatialDimType eq 'COUNTRY' and "
            f"TimeDim ge {YEAR_START} and TimeDim le {YEAR_END}"
        )
    }
    all_rows = []
    while url:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        all_rows.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
        params = None
    return all_rows

# Once sadece HWF_0001 (Hekim Yogunlugu) ile test edelim
test_rows = fetch_indicator("HWF_0001")
print(f"Toplam satir: {len(test_rows)}")
print(test_rows[0])  # ilk satirin tam yapisini gorelim

Toplam satir: 673
{'Id': 9758352, 'IndicatorCode': 'HWF_0001', 'SpatialDimType': 'COUNTRY', 'SpatialDim': 'KGZ', 'ParentLocationCode': 'EUR', 'TimeDimType': 'YEAR', 'ParentLocation': 'Europe', 'Dim1Type': None, 'Dim1': None, 'TimeDim': 2023, 'Dim2Type': None, 'Dim2': None, 'Dim3Type': None, 'Dim3': None, 'DataSourceDimType': None, 'DataSourceDim': None, 'Value': '15.68', 'NumericValue': 15.68, 'Low': None, 'High': None, 'Comments': 'NHWA data portal, December 2025 update; https://apps.who.int/nhwaportal/', 'Date': '2026-01-23T13:29:16.4+01:00', 'TimeDimensionValue': '2023', 'TimeDimensionBegin': '2023-01-01T00:00:00+01:00', 'TimeDimensionEnd': '2023-12-31T00:00:00+01:00'}


In [7]:
import pandas as pd

df_test = pd.DataFrame(test_rows)

# Ayni ulke + yil kombinasyonu birden fazla kez geciyor mu?
dupe_check = df_test.groupby(['SpatialDim', 'TimeDim']).size()
print("Mukerrer olan ulke-yil sayisi:", (dupe_check > 1).sum())
print(dupe_check[dupe_check > 1].head(10))

Mukerrer olan ulke-yil sayisi: 0
Series([], dtype: int64)


In [8]:
year_counts = df_test['TimeDim'].value_counts().sort_index()
print(year_counts)

TimeDim
2019    140
2020    145
2021    133
2022    138
2023    117
Name: count, dtype: int64


In [9]:
hemsire_rows = fetch_indicator("HWF_0006")
print(f"Hemsire-Ebe: {len(hemsire_rows)} satir")

dtp3_rows = fetch_indicator("WHS4_100")
print(f"DTP3: {len(dtp3_rows)} satir")

Hemsire-Ebe: 663 satir
DTP3: 970 satir


In [10]:
# Uc gostergeyi tek listede topla
all_rows = test_rows + hemsire_rows + dtp3_rows

labels_by_code = {
    "HWF_0001": "Hekim Yogunlugu",
    "HWF_0006": "Hemsire-Ebe Yogunlugu",
    "WHS4_100": "DTP3 Asi Orani",
}

out_rows = []
for r in all_rows:
    out_rows.append({
        "Ulke": country_lookup.get(r["SpatialDim"], r["SpatialDim"]),
        "UlkeKodu": r["SpatialDim"],
        "Bolge": r.get("ParentLocation"),   # API zaten tam isim veriyor
        "Yil": r["TimeDim"],
        "Gosterge": labels_by_code[r["IndicatorCode"]],
        "Deger": r.get("NumericValue"),
    })

df_final = pd.DataFrame(out_rows)
print(f"Toplam satir: {len(df_final)}")
print(df_final['Gosterge'].value_counts())
print(f"\nBos deger sayisi: {df_final['Deger'].isna().sum()}")

Toplam satir: 2306
Gosterge
DTP3 Asi Orani           970
Hekim Yogunlugu          673
Hemsire-Ebe Yogunlugu    663
Name: count, dtype: int64

Bos deger sayisi: 0


In [11]:
output_path = "clean/clean_uzun_format_api.csv"
df_final.to_csv(output_path, index=False)
print(f"Kaydedildi: {output_path}")
df_final.head(10)

Kaydedildi: clean/clean_uzun_format_api.csv


,Ulke,UlkeKodu,Bolge,Yil,Gosterge,Deger
0,Kyrgyzstan,KGZ,Europe,2023,Hekim Yogunlugu,15.68
1,Mexico,MEX,Americas,2023,Hekim Yogunlugu,27.52
2,Qatar,QAT,Eastern Mediterranean,2019,Hekim Yogunlugu,27.32
3,Rwanda,RWA,Africa,2019,Hekim Yogunlugu,1.17
4,Botswana,BWA,Africa,2021,Hekim Yogunlugu,4.22
5,Slovakia,SVK,Europe,2020,Hekim Yogunlugu,36.71
6,Costa Rica,CRI,Americas,2020,Hekim Yogunlugu,24.94
7,Indonesia,IDN,Western Pacific,2023,Hekim Yogunlugu,5.24
8,Brunei Darussalam,BRN,Western Pacific,2020,Hekim Yogunlugu,21.19
9,Austria,AUT,Europe,2021,Hekim Yogunlugu,54.02
